# EXIST 2026 — Estrategia principal unificada: **Cross-Attention Texto ↔ Fisiología** (Memes + Vídeos)

Un **único** notebook que aplica la **misma estrategia de fusión por cross-attention** tanto a
**memes (Tarea 2)** como a **vídeos (Tarea 3)** del reto EXIST 2026, optimizando hiperparámetros
con **Optuna (TPE + MedianPruner)**.

Ambas tareas comparten la misma estructura profunda —por eso pueden unificarse:

| | Memes (Tarea 2) | Vídeos (Tarea 3) |
|---|---|---|
| Stream de texto | Captions Qwen3-VL (`[CAP][SAN][OCR]`) | Súper-Texto Qwen3-VL/ASR (`[ANALYSIS][DESC][OCR][ASR]`) |
| Stream fisiológico | EEG + HR + ET (agregado por sujeto) | EEG + HR + ET (matriz por sujeto + attention pool) |
| Subtareas jerárquicas | T2.1 · T2.2 · T2.3 | T3.1 · T3.2 · T3.3 |
| Etiquetas | Soft (LeWiDi) | Soft (LeWiDi) |
| Métrica oficial | ICM-Soft (PyEvALL) | ICM-Soft (PyEvALL) |

## La estrategia principal: fusión por Cross-Attention

```
Texto (tokens)  ──▶ Encoder (Transformer) ──▶ mean-pool ─▶ z_text (B, d_text)
                                                                │
Fisiología ─▶ PhysioEncoder (según modalidad) ─▶ z_phys (B, d_phys)
   EEG/HR/ET       memes: MLP sobre vector agregado                │
                   vídeos: MLP compartido + AttentionPool sujetos  │
                                                ┌───────────────────┴───────────────────┐
                                                │   CrossAttentionFusion (BIDIRECCIONAL) │
                                                │   texto→fisio  Y  fisio→texto          │
                                                │   residual + LayerNorm + concat        │
                                                └───────────────────┬───────────────────┘
                                                                     ▼
                                        ┌────────────────────────────┴───────────────────────────┐
                                        │            UN MODELO POR SUBTAREA (independiente)         │
                                        │  T*.1: 2 logits softmax   [YES, NO]                       │
                                        │  T*.2: 3 logits softmax   [NO, DIRECT, JUDGEMENTAL]       │
                                        │  T*.3: 6 logits sigmoid   [NO, +5 categorías] (multilabel)│
                                        └──────────────────────────────────────────────────────────┘
```

**Por qué cross-attention (y no concatenación):** cada modalidad *reinterpreta* a la otra.
El texto atiende a la reacción fisiológica (arousal alto + texto sospechoso ⇒ refuerza señal de
sexismo sutil) y la fisiología atiende al texto (reacción débil + texto explícito ⇒ relativiza).
La concatenación plana no permite esta modulación cruzada.

## Cómo ejecutar

1. En la celda **CONFIG** elige `MODALITY = "memes"` **o** `MODALITY = "videos"`.
2. Ejecuta el notebook de arriba abajo.
3. Para correr la **otra** modalidad, cambia `MODALITY` y **re-ejecuta** desde CONFIG.

Los checkpoints y submissions se guardan en carpetas separadas por modalidad, así que ambas
ejecuciones no se pisan.

---

## Índice

| Sec | Contenido |
|-----|-----------|
| 0 | Setup (imports, semilla, dispositivo) |
| 1 | **CONFIG** — switch `MODALITY` + rutas y ajustes por modalidad |
| 2 | Carga de datos + construcción del texto (memes/vídeos) |
| 3 | Soft labels LeWiDi (canónicas para ambas modalidades) |
| 4 | Extracción fisiológica (agregada / matriz por sujeto) |
| 5 | Split train/val (group-aware vídeos · estratificado memes) |
| 6 | Preprocesado fisiológico (imputación + z-score sin leakage) |
| 7 | Tokenizer + Dataset unificado |
| 8 | **Modelo cross-attention unificado** (PhysioEncoder + fusión + cabeza) |
| 9 | Pérdidas + evaluación oficial ICM-Soft (PyEvALL) |
| 10 | `train_one_task` (Optuna-aware con pruning) |
| 11 | **Optuna** (TPE + MedianPruner) por subtarea |
| 12 | Entrenamiento final + curvas |
| 13 | Refit 100% + predicciones test (submission) |


## 0 · Setup (imports, semilla, dispositivo)

In [ ]:
# === Imports ===
import os, sys, re, json, math, gc, random
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# Reproducibilidad
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('PyTorch:', torch.__version__, '| Device:', DEVICE)
if DEVICE == 'cuda':
    print('GPU    :', torch.cuda.get_device_name(0))


## 1 · CONFIG — elige la modalidad

**Cambia `MODALITY`** a `"memes"` o `"videos"` y re-ejecuta el notebook. Todo lo demás se
configura automáticamente a partir de esta única variable.

- **`MODALITY = "memes"`** → Tarea 2 (imágenes). Fisiología agregada por sujeto (vector plano),
  encoder de texto bilingüe, subtareas `21/22/23`.
- **`MODALITY = "videos"`** → Tarea 3 (TikTok). Fisiología como matriz por sujeto + attention pool,
  Súper-Texto con ASR, subtareas `31/32/33`.


In [ ]:
# === 1 · CONFIG =============================================================
MODALITY = "videos"   # <<<<<<  cambia a "memes" o "videos"  >>>>>>
assert MODALITY in ("memes", "videos")

# Rutas raíz (Colab / local)
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = Path('/content/drive/MyDrive/EXIST_2026')
else:
    PROJECT_ROOT = Path(r'c:/Users/juana/OneDrive/Escritorio/TFG/EXIST_2026')

# --- Configuración por modalidad -------------------------------------------
if MODALITY == "videos":
    CFG = dict(
        task_ids   = ["31", "32", "33"],
        physio_kind= "matrix",                       # matriz (N, S, F) + attention pool
        text_model = "j-hartmann/emotion-english-roberta-large",
        max_len    = 512,
        train_json = PROJECT_ROOT / 'EXIST 2026 Dataset V0.1' / 'Dataset - Evall'
                      / 'EXIST 2026 Videos Dataset' / 'training' / 'EXIST2026_training.json',
        vlm_json   = PROJECT_ROOT / 'cache_qwen_videos' / 'vlm_video_analysis_qwen3vl_no_urls.json',
        test_json_candidates = [
            PROJECT_ROOT / 'EXIST 2026 Dataset V0.1' / 'Dataset - Evall'
              / 'EXIST 2026 Videos Dataset' / 'test' / 'EXIST2026_test.json',
        ],
        work_dir   = PROJECT_ROOT / 'work_main_strategy' / 'videos',
    )
else:  # memes
    CFG = dict(
        task_ids   = ["21", "22", "23"],
        physio_kind= "flat",                         # vector agregado (N, D)
        text_model = "FacebookAI/xlm-roberta-base",  # bilingüe EN+ES en un solo modelo
        max_len    = 256,
        train_json = PROJECT_ROOT / 'EXIST 2026 Dataset V0.1' / 'Dataset - Evall'
                      / 'EXIST 2026 Memes Dataset' / 'training' / 'EXIST2026_training_memes.json',
        vlm_json   = PROJECT_ROOT / 'EXIST 2026 Dataset V0.1' / 'vlm_captions'
                      / 'vlm_captions_memes.json',
        test_json_candidates = [
            PROJECT_ROOT / 'EXIST 2026 Dataset V0.1' / 'Dataset - Evall'
              / 'EXIST 2026 Memes Dataset' / 'test' / 'EXIST2026_test_memes.json',
        ],
        work_dir   = PROJECT_ROOT / 'work_main_strategy' / 'memes',
    )

TASK_IDS   = CFG["task_ids"]
PHYSIO_KIND= CFG["physio_kind"]
MAX_LEN    = CFG["max_len"]
WORK_DIR   = CFG["work_dir"]
(WORK_DIR / 'ckpt').mkdir(parents=True, exist_ok=True)
(WORK_DIR / 'figs').mkdir(parents=True, exist_ok=True)
(WORK_DIR / 'submissions').mkdir(parents=True, exist_ok=True)

# Categorías de la subtarea 3 (idénticas en ambas modalidades)
T3_CATS = ['IDEOLOGICAL-INEQUALITY', 'STEREOTYPING-DOMINANCE', 'OBJECTIFICATION',
           'SEXUAL-VIOLENCE', 'MISOGYNY-NON-SEXUAL-VIOLENCE']

# Helpers de subtarea: el último dígito define el tipo.
def task_kind(t):
    """'bin' (T*.1) | 'tri' (T*.2) | 'multi' (T*.3)."""
    return {'1': 'bin', '2': 'tri', '3': 'multi'}[t[-1]]

TASK_NOUT = {t: {'bin': 2, 'tri': 3, 'multi': 6}[task_kind(t)] for t in TASK_IDS}

print(f"MODALITY = {MODALITY}")
print(f"  subtareas : {TASK_IDS}  (n_out={TASK_NOUT})")
print(f"  physio    : {PHYSIO_KIND}")
print(f"  encoder   : {CFG['text_model']}")
print(f"  work_dir  : {WORK_DIR}")
print(f"  train_json: {CFG['train_json']}")
print(f"  train existe: {CFG['train_json'].exists()}")


## 2 · Carga de datos + construcción del texto

Cargamos el JSON de entrenamiento y el JSON de captions/análisis de Qwen3-VL. Definimos
`build_text(id)` que produce el documento textual de cada instancia:

- **Vídeos**: Súper-Texto `[ANALYSIS] {sexism_analysis} [DESC] {visual_description} [OCR] {ocr} [ASR] {asr}`.
- **Memes**: `{visual_description} [SEP] {social_analysis} [SEP] {ocr}` (config `capsanocr`).


In [ ]:
# === 2 · Carga de datos + build_text ========================================
with open(CFG["train_json"], "r", encoding="utf-8", errors="replace") as f:
    DATA = json.load(f)
print(f"Instancias de entrenamiento: {len(DATA):,}")

VLM = {}
if CFG["vlm_json"].exists():
    with open(CFG["vlm_json"], "r", encoding="utf-8") as f:
        _vlm_raw = json.load(f)
    # Puede venir como lista de dicts o dict {id: ...}
    if isinstance(_vlm_raw, list):
        VLM = {str(r.get("id_EXIST")): r for r in _vlm_raw}
    else:
        VLM = {str(k): v for k, v in _vlm_raw.items()}
    print(f"VLM captions: {len(VLM):,}")
else:
    print(f"[!] No existe VLM JSON ({CFG['vlm_json'].name}); el texto usará solo OCR del dataset.")


def _clean(s):
    return (s or "").strip() if isinstance(s, str) else ""

SEP = " [SEP] "

def build_text(inst, vlm_entry):
    """Documento textual de una instancia según la modalidad."""
    if MODALITY == "videos":
        parsed = (vlm_entry or {}).get("qwen_parsed") or {}
        parsed = parsed if isinstance(parsed, dict) else {}
        analysis = _clean(parsed.get("sexism_analysis"))
        desc     = _clean(parsed.get("visual_description"))
        ocr      = _clean(parsed.get("ocr_text")) or _clean(inst.get("text"))
        asr      = _clean((vlm_entry or {}).get("asr_transcript"))
        return f"[ANALYSIS] {analysis} [DESC] {desc} [OCR] {ocr} [ASR] {asr}"
    else:  # memes — capsanocr
        e   = vlm_entry or {}
        cap = _clean(e.get("visual_description"))
        san = _clean(e.get("sexism_analysis") or e.get("social_analysis"))
        ocr = _clean(inst.get("text"))
        parts = [p for p in (cap, san, ocr) if p]
        return SEP.join(parts) if parts else ""

# Ejemplo
_any = next(iter(DATA))
print("\n--- Texto de muestra ---")
print(build_text(DATA[_any], VLM.get(_any))[:500], "...")


## 3 · Soft labels (LeWiDi) — convención canónica

Bajo *Learning with Disagreement* entrenamos sobre la **proporción de anotadores** por clase.
Unificamos el orden de las clases entre modalidades:

- **T*.1** → `[P(YES), P(NO)]`
- **T*.2** → `[P(NO), P(DIRECT), P(JUDGEMENTAL)]` (el `-` cuenta como NO)
- **T*.3** → `[P(NO), P(IDEO), P(STER), P(OBJ), P(SEX), P(MIS)]` (multi-label, no normaliza)

`UNKNOWN` se ignora siempre.


In [ ]:
# === 3 · Soft labels canónicas =============================================
def _labels(inst, n):
    """Devuelve la lista de anotaciones de la subtarea n (1/2/3) de la instancia,
    tolerante a la variante con espacio 'labels_ task2_2'."""
    suffix = TASK_IDS[n - 1]   # '31'|'32'|'33' o '21'|'22'|'23'
    t = suffix[-1]
    grp = suffix[0]            # '3' videos, '2' memes
    for key in (f"labels_task{grp}_{t}", f"labels_ task{grp}_{t}"):
        if key in inst:
            return inst[key] or []
    return []


def soft_bin(labs):
    valid = [x for x in labs if x != "UNKNOWN"]
    if not valid:
        return None
    p_yes = sum(1 for x in valid if x == "YES") / len(valid)
    return np.array([p_yes, 1.0 - p_yes], dtype=np.float32)   # [YES, NO]


def soft_tri(labs):
    valid = [x for x in labs if x != "UNKNOWN"]
    if not valid:
        return np.array([1.0, 0.0, 0.0], dtype=np.float32)
    n = len(valid)
    p_no   = sum(1 for x in valid if x == "-")           / n
    p_dir  = sum(1 for x in valid if x == "DIRECT")      / n
    p_judg = sum(1 for x in valid if x == "JUDGEMENTAL") / n
    return np.array([p_no, p_dir, p_judg], dtype=np.float32)  # [NO, DIR, JUDG]


def soft_multi(arrs):
    valid = [a for a in arrs if "UNKNOWN" not in a]
    n = len(valid)
    if n == 0:
        return None
    out = np.zeros(6, dtype=np.float32)
    out[0] = sum(1 for a in valid if a == ["-"]) / n         # P(NO)
    for i, cat in enumerate(T3_CATS):
        out[i + 1] = sum(1 for a in valid if cat in a) / n   # P(cat)
    return out


def build_soft(inst):
    """{ taskid: vector } o None si no hay etiquetas válidas en T*.1/T*.3."""
    t1, t2, t3 = TASK_IDS
    v1 = soft_bin(_labels(inst, 1))
    v3 = soft_multi(_labels(inst, 3))
    if v1 is None or v3 is None:
        return None
    return {t1: v1, t2: soft_tri(_labels(inst, 2)), t3: v3}


y_all, _drop = {}, 0
for k, inst in DATA.items():
    y = build_soft(inst)
    if y is None:
        _drop += 1
    else:
        y_all[str(k)] = y
print(f"Soft labels válidas: {len(y_all):,}/{len(DATA):,} (descartadas={_drop})")
_ex = next(iter(y_all))
for t in TASK_IDS:
    print(f"  {t} ({task_kind(t)}): {np.round(y_all[_ex][t], 3).tolist()}")


## 4 · Extracción fisiológica (EEG + HR + ET)

El vector fisiológico se construye distinto según la modalidad, pero **ambos alimentan la misma
fusión cross-attention**:

- **Vídeos (`matrix`)**: matriz `(MAX_SUBJECTS, 108)` por vídeo (features crudas por sujeto) +
  máscara de sujetos válidos. El modelo la resume con un MLP compartido + *attention pool* sobre
  sujetos (permutation-invariant).
- **Memes (`flat`)**: vector agregado por meme — para cada feature, estadísticos `(mean, std, min, max)`
  entre sujetos. Produce un vector plano de dimensión fija.

Guardamos la fisiología en un dict `PHYSIO_RAW = {id: array}` (y `PHYSIO_MASK` para vídeos) para
desacoplar la extracción del preprocesado y el `Dataset`.


In [ ]:
# === 4 · Inventario de features + extracción ================================
# Descubrimos el schema canónico de features escaneando DATA (orden estable).
_feat = {"ET": set(), "HR": set(), "EEG": set()}
for _inst in DATA.values():
    _mods = (_inst.get("sensorial", {}) or {}).get("modalities", {}) or {}
    for _mod, _payload in _mods.items():
        if _mod not in _feat:
            continue
        for _ud in (_payload.get("by_user", {}) or {}).values():
            if isinstance(_ud, dict):
                _feat[_mod].update(_ud.keys())

FEAT_ET  = sorted(_feat["ET"]);  FEAT_HR = sorted(_feat["HR"]);  FEAT_EEG = sorted(_feat["EEG"])
FEAT_ALL = FEAT_ET + FEAT_HR + FEAT_EEG
MOD_OF   = np.array(["ET"]*len(FEAT_ET) + ["HR"]*len(FEAT_HR) + ["EEG"]*len(FEAT_EEG))
FEAT_DIM = len(FEAT_ALL)
MAX_SUBJECTS = 4
PHYSIO_AGGS  = ("mean", "std", "min", "max")
print(f"Features: ET={len(FEAT_ET)} HR={len(FEAT_HR)} EEG={len(FEAT_EEG)} | TOTAL={FEAT_DIM}")


def _sensor_matrix(inst):
    """Vídeos: (MAX_SUBJECTS, FEAT_DIM) + máscara (MAX_SUBJECTS,)."""
    sens = inst.get("sensorial", {}) or {}
    users = (sens.get("users", []) or [])[:MAX_SUBJECTS]
    mods  = sens.get("modalities", {}) or {}
    X = np.full((MAX_SUBJECTS, FEAT_DIM), np.nan, dtype=np.float32)
    mask = np.zeros(MAX_SUBJECTS, dtype=np.float32)
    for i, u in enumerate(users):
        for j, fname in enumerate(FEAT_ALL):
            mod = MOD_OF[j]
            v = ((mods.get(mod, {}) or {}).get("by_user", {}) or {}).get(u, {}).get(fname, np.nan)
            if v is not None:
                try: X[i, j] = float(v)
                except (ValueError, TypeError): pass
        mask[i] = 1.0
    X = np.where(np.isnan(X) & (mask[:, None] == 0), 0.0, X)
    return X, mask


def _flat_vector(inst, modalities=("EEG", "HR", "ET")):
    """Memes: vector agregado (mean/std/min/max entre sujetos) por modalidad."""
    import warnings
    mods = (inst.get("sensorial", {}) or {}).get("modalities", {}) or {}
    feat_by_mod = {"ET": FEAT_ET, "HR": FEAT_HR, "EEG": FEAT_EEG}
    parts = []
    for mod in modalities:
        feats = feat_by_mod[mod]
        by_user = (mods.get(mod, {}) or {}).get("by_user", {}) or {}
        n_dims = len(feats) * len(PHYSIO_AGGS)
        if not by_user or not feats:
            parts.append(np.zeros(n_dims, dtype=np.float32)); continue
        users = list(by_user.keys())
        X = np.full((len(users), len(feats)), np.nan, dtype=np.float32)
        for i, u in enumerate(users):
            ud = by_user[u]
            if not isinstance(ud, dict): continue
            for j, fn in enumerate(feats):
                v = ud.get(fn)
                if v is not None:
                    try: X[i, j] = float(v)
                    except (ValueError, TypeError): pass
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            for agg in PHYSIO_AGGS:
                v = {"mean": np.nanmean, "std": np.nanstd,
                     "min": np.nanmin, "max": np.nanmax}[agg](X, axis=0)
                parts.append(np.where(np.isfinite(v), v, 0.0).astype(np.float32))
    return np.concatenate(parts) if parts else np.zeros(0, dtype=np.float32)


# Construimos la fisiología por instancia
IDS_ALL = list(y_all.keys())
PHYSIO_RAW, PHYSIO_MASK = {}, {}
for k in IDS_ALL:
    if PHYSIO_KIND == "matrix":
        X, m = _sensor_matrix(DATA[k]); PHYSIO_RAW[k] = X; PHYSIO_MASK[k] = m
    else:
        PHYSIO_RAW[k] = _flat_vector(DATA[k])

if PHYSIO_KIND == "matrix":
    PHYSIO_IN_DIM = FEAT_DIM
    print(f"Physio (matrix): {len(PHYSIO_RAW)} × ({MAX_SUBJECTS}, {FEAT_DIM})")
else:
    PHYSIO_IN_DIM = PHYSIO_RAW[IDS_ALL[0]].shape[0]
    print(f"Physio (flat)  : {len(PHYSIO_RAW)} × ({PHYSIO_IN_DIM},)")


## 5 · Split train/val

- **Vídeos**: *group-aware* por creador de TikTok (extraído de la URL) — el mismo creador nunca
  aparece en train y val a la vez, evitando fuga por estilo.
- **Memes**: estratificado por `(idioma, voto mayoritario de T*.1)`.


In [ ]:
# === 5 · Split train/val ====================================================
rows = []
for k in IDS_ALL:
    inst = DATA[k]
    p_yes = float(y_all[k][TASK_IDS[0]][0])
    if MODALITY == "videos":
        mobj = re.search(r'@([\w_\.]+)', inst.get("url", "") or "")
        group = mobj.group(1) if mobj else f"__solo_{k}"
    else:
        group = k   # memes: cada uno su grupo (split por instancia)
    rows.append({"id": k, "lang": inst.get("lang", "en"),
                 "group": group, "p_yes": p_yes,
                 "hard_yes": int(p_yes >= 0.5)})
dfm = pd.DataFrame(rows)

if MODALITY == "videos":
    # split por creador, estratificado por su tasa media de YES
    gstats = dfm.groupby("group").agg(p_yes=("p_yes", "mean")).reset_index()
    gstats["strat"] = pd.qcut(gstats.p_yes.rank(method="first"), q=4, labels=False)
    tr_g, vl_g = train_test_split(gstats.group, test_size=0.15,
                                   random_state=SEED, stratify=gstats.strat)
    dfm["split"] = np.where(dfm.group.isin(set(tr_g)), "train", "val")
else:
    # split por instancia, estratificado por (lang, hard_yes)
    dfm["strat"] = dfm.lang.astype(str) + "_" + dfm.hard_yes.astype(str)
    tr_i, vl_i = train_test_split(dfm.index, test_size=0.15,
                                   random_state=SEED, stratify=dfm.strat)
    dfm["split"] = "train"; dfm.loc[vl_i, "split"] = "val"

ids_train = dfm[dfm.split == "train"].id.tolist()
ids_val   = dfm[dfm.split == "val"].id.tolist()
print("Split:", dfm.split.value_counts().to_dict())
print("Balance YES por split:\n", dfm.groupby("split").hard_yes.mean().round(3))
print("Idiomas por split:\n", pd.crosstab(dfm.split, dfm.lang))


## 6 · Preprocesado fisiológico (sin leakage)

Los estadísticos se ajustan **solo con train**:

- **Vídeos**: imputación KNN (k=5) sobre filas de sujeto válidas → z-score por modalidad (ET/HR/EEG).
- **Memes**: z-score global del vector agregado (la extracción ya rellenó ausencias con 0).

Guardamos las transformaciones en `PREPROC` para re-aplicarlas al test en la sección 13.


In [ ]:
# === 6 · Preprocesado fisiológico ==========================================
PREPROC = {}

if PHYSIO_KIND == "matrix":
    Xtr = np.stack([PHYSIO_RAW[k] for k in ids_train])   # (Ntr, S, F)
    Mtr = np.stack([PHYSIO_MASK[k] for k in ids_train])
    valid = Mtr.reshape(-1) > 0
    knn = KNNImputer(n_neighbors=5, weights="distance")
    knn.fit(Xtr.reshape(-1, FEAT_DIM)[valid])

    def _impute(X):
        return knn.transform(X.reshape(-1, FEAT_DIM)).reshape(X.shape)

    Xtr_imp = _impute(Xtr)
    scalers = {}
    for mod in ("ET", "HR", "EEG"):
        cols = np.where(MOD_OF == mod)[0]
        sc = StandardScaler().fit(Xtr_imp.reshape(-1, FEAT_DIM)[valid][:, cols])
        scalers[mod] = sc

    def _standardize(X):
        flat = X.reshape(-1, FEAT_DIM).copy()
        for mod in ("ET", "HR", "EEG"):
            cols = np.where(MOD_OF == mod)[0]
            flat[:, cols] = scalers[mod].transform(flat[:, cols])
        return flat.reshape(X.shape).astype(np.float32)

    def physio_transform(k):
        """Aplica impute+z a la fisiología cruda de la instancia k. Devuelve (X_z, mask)."""
        X = PHYSIO_RAW[k][None]; return _standardize(_impute(X))[0], PHYSIO_MASK[k]

    PREPROC = {"knn": knn, "scalers": scalers}
    # Sanity
    _z, _m = physio_transform(ids_train[0])
    print(f"physio matrix z-scored ok: {_z.shape}, NaN={int(np.isnan(_z).sum())}")

else:  # flat
    Xtr = np.stack([PHYSIO_RAW[k] for k in ids_train])   # (Ntr, D)
    mu    = Xtr.mean(axis=0).astype(np.float32)
    sigma = (Xtr.std(axis=0) + 1e-8).astype(np.float32)

    def physio_transform(k):
        return ((PHYSIO_RAW[k] - mu) / sigma).astype(np.float32), None

    PREPROC = {"mu": mu, "sigma": sigma}
    _z, _ = physio_transform(ids_train[0])
    print(f"physio flat z-scored ok: {_z.shape}")


## 7 · Tokenizer + Dataset unificado

Un único `ExistDataset` sirve a ambas modalidades: tokeniza el texto y adjunta la fisiología ya
transformada. El `collate` apila `physio` (y `physio_mask` solo en vídeos) según `PHYSIO_KIND`.


In [ ]:
# === 7 · Tokenizer + Dataset ================================================
tokenizer = AutoTokenizer.from_pretrained(CFG["text_model"])
print("Tokenizer:", tokenizer.__class__.__name__, "| vocab:", tokenizer.vocab_size)


class ExistDataset(Dataset):
    """Texto tokenizado + fisiología transformada + soft label de una subtarea.

    Si `task` es None, incluye las soft labels de las 3 subtareas (para eval jerárquica);
    si se indica, solo la de esa subtarea (entrenamiento single-task)."""
    def __init__(self, ids, task=None, max_len=MAX_LEN):
        self.ids = ids; self.task = task; self.max_len = max_len
    def __len__(self): return len(self.ids)
    def __getitem__(self, idx):
        k = self.ids[idx]
        enc = tokenizer(build_text(DATA[k], VLM.get(k)), truncation=True,
                        max_length=self.max_len, padding="max_length", return_tensors="pt")
        Xz, mask = physio_transform(k)
        item = {
            "id_EXIST": k,
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "physio": torch.from_numpy(np.ascontiguousarray(Xz)),
        }
        if PHYSIO_KIND == "matrix":
            item["physio_mask"] = torch.from_numpy(np.ascontiguousarray(mask))
        # soft labels
        for t in TASK_IDS:
            item[f"y_{t}"] = torch.from_numpy(y_all[k][t])
        return item


def collate(batch):
    out = {"id_EXIST": [b["id_EXIST"] for b in batch]}
    keys = ["input_ids", "attention_mask", "physio"] + [f"y_{t}" for t in TASK_IDS]
    if PHYSIO_KIND == "matrix":
        keys.append("physio_mask")
    for k in keys:
        out[k] = torch.stack([b[k] for b in batch], dim=0)
    return out


# Sanity: mini-batch
_ds = ExistDataset(ids_train)
_demo = collate([_ds[i] for i in range(2)])
for k, v in _demo.items():
    print(f"  {k:14s} {tuple(v.shape) if isinstance(v, torch.Tensor) else v}")


## 8 · Modelo cross-attention unificado

La pieza central. Tres bloques:

1. **Encoder de texto** (Transformer congelado por defecto) → `z_text` (mean-pool sobre tokens válidos).
2. **PhysioEncoder** — polimórfico según `PHYSIO_KIND`:
   - `flat` (memes): `LayerNorm → MLP` sobre el vector agregado.
   - `matrix` (vídeos): `SharedSubjectMLP` (mismos pesos para todos los sujetos) + `AttentionPool`
     (colapsa sujetos con pesos aprendidos, permutation-invariant).
3. **CrossAttentionFusion** (idéntica para ambas modalidades) — atención cruzada **bidireccional**
   texto↔fisiología con residual + LayerNorm, y concatenación final.

Se crea **un modelo independiente por subtarea** (`n_out` = 2 / 3 / 6).


In [ ]:
# === 8.1 · Componentes fisiológicos =========================================
class _SharedSubjectMLP(nn.Module):
    """MLP compartido (mismos pesos para todo sujeto): 108 features -> embedding."""
    def __init__(self, in_dim, hidden=256, out_dim=128, p_drop=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.GELU(), nn.Dropout(p_drop),
            nn.Linear(hidden, hidden), nn.GELU(), nn.Dropout(p_drop),
            nn.Linear(hidden, out_dim),
        )
    def forward(self, x):                    # x: (B, S, F)
        B, S, Fdim = x.shape
        return self.net(x.reshape(B * S, Fdim)).reshape(B, S, -1)


class _AttentionPool(nn.Module):
    """Pooling por atencion sobre sujetos (permutation-invariant)."""
    def __init__(self, dim):
        super().__init__()
        self.query = nn.Parameter(torch.randn(dim) / math.sqrt(dim))
        self.proj = nn.Linear(dim, dim)
    def forward(self, x, mask):              # x: (B, S, D), mask: (B, S)
        scores = (torch.tanh(self.proj(x)) * self.query).sum(-1)
        scores = scores.masked_fill(mask < 0.5, float("-inf"))
        w = F.softmax(scores, dim=-1)
        w = torch.nan_to_num(w, nan=1.0 / x.size(1))
        return (x * w.unsqueeze(-1)).sum(1), w


class PhysioEncoder(nn.Module):
    """Front-end fisiologico polimorfico -> vector (B, out_dim).
       kind='flat'  : MLP sobre vector agregado (memes).
       kind='matrix': SharedSubjectMLP + AttentionPool (videos)."""
    def __init__(self, kind, in_dim, hidden=256, out_dim=128, p_drop=0.2):
        super().__init__()
        self.kind = kind; self.out_dim = out_dim
        if kind == "flat":
            self.net = nn.Sequential(
                nn.LayerNorm(in_dim),
                nn.Linear(in_dim, hidden), nn.GELU(), nn.Dropout(p_drop),
                nn.Linear(hidden, out_dim),
            )
        else:
            self.mlp  = _SharedSubjectMLP(in_dim, hidden, out_dim, p_drop)
            self.pool = _AttentionPool(out_dim)
    def forward(self, batch):
        if self.kind == "flat":
            return self.net(batch["physio"]), None
        e = self.mlp(batch["physio"])
        return self.pool(e, batch["physio_mask"])


# === 8.2 · Fusion cross-attention (bidireccional) ===========================
class CrossAttentionFusion(nn.Module):
    """Atencion cruzada texto<->fisiologia. Proyecta ambos a `hidden`, cruza en las 2
    direcciones (texto->fisio, fisio->texto), residual+LayerNorm y concatena -> (B, 2*hidden)."""
    def __init__(self, text_dim, phys_dim, hidden=512, num_heads=8, p_drop=0.1):
        super().__init__()
        self.text_proj = nn.Linear(text_dim, hidden)
        self.phys_proj = nn.Linear(phys_dim, hidden)
        self.t2p = nn.MultiheadAttention(hidden, num_heads, dropout=p_drop, batch_first=True)
        self.p2t = nn.MultiheadAttention(hidden, num_heads, dropout=p_drop, batch_first=True)
        self.norm_t = nn.LayerNorm(hidden)
        self.norm_p = nn.LayerNorm(hidden)
        self.dropout = nn.Dropout(p_drop)
        self.out_dim = 2 * hidden
    def forward(self, text_vec, phys_vec):
        t = self.text_proj(text_vec).unsqueeze(1)
        p = self.phys_proj(phys_vec).unsqueeze(1)
        t2p, _ = self.t2p(t, p, p)
        p2t, _ = self.p2t(p, t, t)
        t_f = self.norm_t(t + self.dropout(t2p)).squeeze(1)
        p_f = self.norm_p(p + self.dropout(p2t)).squeeze(1)
        return torch.cat([t_f, p_f], dim=-1)


In [ ]:
# === 8.3 · Modelo por subtarea (factory) ====================================
def make_model_cls(n_out):
    class _Model(nn.Module):
        def __init__(self, text_model=CFG["text_model"], physio_in=PHYSIO_IN_DIM,
                     sensor_hidden=256, sensor_out=128, fusion_hidden=512,
                     num_heads=8, p_drop=0.2, freeze_backbone=True):
            super().__init__()
            self.text_encoder = AutoModel.from_pretrained(text_model)
            text_dim = self.text_encoder.config.hidden_size
            if freeze_backbone:
                for p in self.text_encoder.parameters():
                    p.requires_grad = False
            self.physio = PhysioEncoder(PHYSIO_KIND, physio_in,
                                        sensor_hidden, sensor_out, p_drop)
            self.fusion = CrossAttentionFusion(text_dim, sensor_out,
                                               fusion_hidden, num_heads, p_drop)
            self.dropout = nn.Dropout(p_drop)
            self.head = nn.Linear(self.fusion.out_dim, n_out)
            self.n_out = n_out
        @staticmethod
        def _mean_pool(last_hidden, attn):
            m = attn.unsqueeze(-1).float()
            return (last_hidden * m).sum(1) / m.sum(1).clamp(min=1e-6)
        def forward(self, batch):
            out = self.text_encoder(input_ids=batch["input_ids"],
                                    attention_mask=batch["attention_mask"])
            z_text = self._mean_pool(out.last_hidden_state, batch["attention_mask"])
            z_phys, alphas = self.physio(batch)
            fused = self.dropout(self.fusion(z_text, z_phys))
            return {"logits": self.head(fused), "alphas": alphas}
    _Model.__name__ = f"CrossAttnModel_n{n_out}"
    return _Model

MODEL_CLS = {t: make_model_cls(TASK_NOUT[t]) for t in TASK_IDS}

_demo = MODEL_CLS[TASK_IDS[0]](freeze_backbone=True).to(DEVICE)
_ntr = sum(p.numel() for p in _demo.parameters() if p.requires_grad)
print(f"Modelo {TASK_IDS[0]} OK | entrenables: {_ntr/1e6:.2f}M | fusion out_dim={_demo.fusion.out_dim}")
del _demo; gc.collect()


## 9 · Pérdidas + evaluación oficial ICM-Soft (PyEvALL)

- **Pérdidas** (soft LeWiDi): `T*.1`/`T*.2` → **KL-divergence**; `T*.3` → **BCE** (multilabel).
- **Métrica oficial**: **ICM-Soft / ICM-Soft-Norm** vía PyEvALL sobre el val COMPLETO con
  **gating jerárquico** (si T*.1 predice NO, se fuerza NO en T*.2/T*.3).


In [ ]:
# === 9.1 · Perdidas por subtarea ============================================
def loss_for(task):
    kind = task_kind(task); ykey = f"y_{task}"
    if kind in ("bin", "tri"):
        def _kl(out, batch):
            return F.kl_div(F.log_softmax(out["logits"], -1), batch[ykey], reduction="batchmean")
        return _kl
    def _bce(out, batch):
        return F.binary_cross_entropy_with_logits(out["logits"], batch[ykey])
    return _bce

LOSS_FN = {t: loss_for(t) for t in TASK_IDS}

@torch.no_grad()
def evaluate_proxy(model, loader, task):
    model.eval()
    kind = task_kind(task); ykey = f"y_{task}"; yt, yp = [], []
    for batch in loader:
        b = {k: (v.to(DEVICE) if isinstance(v, torch.Tensor) else v) for k, v in batch.items()}
        logits = model(b)["logits"]
        if kind == "bin":
            yp.append((F.softmax(logits, -1)[:, 0] >= 0.5).cpu().numpy())
            yt.append((b[ykey][:, 0] >= 0.5).cpu().numpy())
        elif kind == "tri":
            yp.append(logits.argmax(-1).cpu().numpy()); yt.append(b[ykey].argmax(-1).cpu().numpy())
        else:
            yp.append((torch.sigmoid(logits) >= 0.5).cpu().numpy()); yt.append((b[ykey] >= 0.5).cpu().numpy())
    yt = np.concatenate(yt); yp = np.concatenate(yp)
    return f1_score(yt, yp, average=("binary" if kind == "bin" else "macro"), zero_division=0)


In [ ]:
# === 9.2 · PyEvALL: instalacion + gold + prediccion + parser ================
try:
    import pyevall  # noqa
except ImportError:
    import subprocess as _sp
    print("Instalando PyEvALL..."); _sp.check_call([sys.executable, "-m", "pip", "install", "-q", "pyevall"])
    import pyevall  # noqa
try:
    from pyevall.evaluation import PyEvALLEvaluation
    from pyevall.utils.utils import PyEvALLUtils
except ImportError:
    from pyevall.evaluation import PyEvALLEvaluation
    from pyevall.utils.utils_py import PyEvALLUtils

PYEVALL_TEST_CASE = "EXIST2025"
TASK_HIER = {"bin": None,
             "tri": {"YES": ["DIRECT", "JUDGEMENTAL"], "NO": []},
             "multi": {"YES": list(T3_CATS), "NO": []}}

def _gold_value(task, vec):
    kind = task_kind(task)
    if kind == "bin":
        return {"YES": float(vec[0]), "NO": float(vec[1])}
    if kind == "tri":
        return {"NO": float(vec[0]), "DIRECT": float(vec[1]), "JUDGEMENTAL": float(vec[2])}
    return {"NO": float(vec[0]), "IDEOLOGICAL-INEQUALITY": float(vec[1]),
            "STEREOTYPING-DOMINANCE": float(vec[2]), "OBJECTIFICATION": float(vec[3]),
            "SEXUAL-VIOLENCE": float(vec[4]), "MISOGYNY-NON-SEXUAL-VIOLENCE": float(vec[5])}

def build_gold(ids, task):
    return [{"test_case": PYEVALL_TEST_CASE, "id": str(k),
             "value": _gold_value(task, y_all[k][task])} for k in ids]

@torch.no_grad()
def predict_pyevall(task, model, m_gate, loader, hierarchical_gate=True):
    model.eval()
    if m_gate is not None and m_gate is not model: m_gate.eval()
    kind = task_kind(task); pred = []
    for batch in loader:
        b = {k: (v.to(DEVICE) if isinstance(v, torch.Tensor) else v) for k, v in batch.items()}
        o = model(b)
        if kind == "bin":
            p = F.softmax(o["logits"], -1).cpu().numpy()
            for i, vid in enumerate(b["id_EXIST"]):
                pred.append({"test_case": PYEVALL_TEST_CASE, "id": str(vid),
                             "value": {"YES": float(p[i, 0]), "NO": float(p[i, 1])}})
            continue
        gate = m_gate if m_gate is not None else model
        pg = F.softmax(gate(b)["logits"], -1).cpu().numpy()
        if kind == "tri":
            pp = F.softmax(o["logits"], -1).cpu().numpy()
            for i, vid in enumerate(b["id_EXIST"]):
                if hierarchical_gate and pg[i, 1] > pg[i, 0]:
                    v = {"NO": 1.0, "DIRECT": 0.0, "JUDGEMENTAL": 0.0}
                else:
                    v = {"NO": float(pp[i, 0]), "DIRECT": float(pp[i, 1]), "JUDGEMENTAL": float(pp[i, 2])}
                pred.append({"test_case": PYEVALL_TEST_CASE, "id": str(vid), "value": v})
        else:
            pp = torch.sigmoid(o["logits"]).cpu().numpy()
            for i, vid in enumerate(b["id_EXIST"]):
                if hierarchical_gate and pg[i, 1] > pg[i, 0]:
                    v = {"NO": 1.0, "IDEOLOGICAL-INEQUALITY": 0.0, "STEREOTYPING-DOMINANCE": 0.0,
                         "OBJECTIFICATION": 0.0, "SEXUAL-VIOLENCE": 0.0,
                         "MISOGYNY-NON-SEXUAL-VIOLENCE": 0.0}
                else:
                    v = {"NO": float(pp[i, 0]), "IDEOLOGICAL-INEQUALITY": float(pp[i, 1]),
                         "STEREOTYPING-DOMINANCE": float(pp[i, 2]), "OBJECTIFICATION": float(pp[i, 3]),
                         "SEXUAL-VIOLENCE": float(pp[i, 4]), "MISOGYNY-NON-SEXUAL-VIOLENCE": float(pp[i, 5])}
                pred.append({"test_case": PYEVALL_TEST_CASE, "id": str(vid), "value": v})
    return pred

def _walk_metric(d, name, depth=0, mx=8):
    if depth > mx: return None
    if isinstance(d, dict):
        if name in d:
            v = d[name]
            if isinstance(v, (int, float)): return float(v)
            if isinstance(v, dict):
                nums = [x for x in v.values() if isinstance(x, (int, float))]
                if nums: return sum(nums) / len(nums)
        for v in d.values():
            r = _walk_metric(v, name, depth + 1, mx)
            if r is not None: return r
    elif isinstance(d, list):
        nums = [r for r in (_walk_metric(x, name, depth + 1, mx) for x in d) if r is not None]
        if nums: return sum(nums) / len(nums)
    return None

def _parse_report(report, metrics=("ICMSoft", "ICMSoftNorm")):
    if report is None: return {}
    out = {}
    for attr in ("report", "dict_report", "json_report", "metrics_data", "data",
                 "result", "results", "_report"):
        raw = getattr(report, attr, None)
        if raw is None: continue
        if hasattr(raw, "columns"):
            for m in metrics:
                if m in raw.columns and m not in out:
                    vals = raw[m].dropna()
                    if len(vals): out[m] = float(vals.mean())
        else:
            for m in metrics:
                if m in out: continue
                v = _walk_metric(raw, m)
                if v is not None: out[m] = v
    if all(m in out for m in metrics): return out
    import io as _io, contextlib as _ctx, re as _re
    buf = _io.StringIO()
    with _ctx.redirect_stdout(buf):
        for fn in ("print_report_tsv", "print_report"):
            try: getattr(report, fn)(); break
            except Exception: continue
    text = buf.getvalue()
    for m in metrics:
        if m in out: continue
        mt = _re.search(rf"\b{m}\b\s*[:=]\s*(-?\d+\.\d+)", text) or \
             _re.search(rf"\b{m}\b[^\d-]{{0,100}}(-?\d+\.\d+)", text)
        if mt:
            try: out[m] = float(mt.group(1))
            except ValueError: pass
    return out

def evaluate_pyevall(task, model, m_gate, loader, gold_list, hierarchical_gate=True):
    pred = predict_pyevall(task, model, m_gate, loader, hierarchical_gate)
    pp = WORK_DIR / "ckpt" / f"_tmp_pred_{task}.json"
    gp = WORK_DIR / "ckpt" / f"_tmp_gold_{task}.json"
    json.dump(pred, open(pp, "w", encoding="utf-8"), ensure_ascii=False)
    json.dump(gold_list, open(gp, "w", encoding="utf-8"), ensure_ascii=False)
    params = {PyEvALLUtils.PARAM_REPORT: PyEvALLUtils.PARAM_OPTION_REPORT_DATAFRAME}
    h = TASK_HIER[task_kind(task)]
    if h is not None: params[PyEvALLUtils.PARAM_HIERARCHY] = h
    report = PyEvALLEvaluation().evaluate(str(pp), str(gp), ["ICMSoft", "ICMSoftNorm"], **params)
    return _parse_report(report, ("ICMSoft", "ICMSoftNorm"))

GOLD_VAL = {t: build_gold(ids_val, t) for t in TASK_IDS}
loader_val_full = DataLoader(ExistDataset(ids_val), batch_size=16, shuffle=False,
                             num_workers=0, collate_fn=collate, pin_memory=(DEVICE == "cuda"))
print("PyEvALL listo. Gold val:", {t: len(GOLD_VAL[t]) for t in TASK_IDS})


## 10 · `train_one_task` — entrenamiento de una subtarea (Optuna-aware)

Entrena **un modelo independiente** por subtarea y selecciona el **mejor checkpoint por ICM-Soft-Norm**
(val COMPLETO con gating jerárquico). Es *Optuna-aware*: `hp` sobreescribe hiperparámetros y `trial`
activa el reporte por época + poda con `MedianPruner`.

**Filtro jerárquico**: T*.2/T*.3 se entrenan solo con instancias YES (`p_yes ≥ 0.5`) pero se evalúan
sobre el val completo con el gate de T*.1.


In [ ]:
# === 10.1 · Subconjuntos YES + defaults =====================================
THR_YES = 0.5
t1 = TASK_IDS[0]
ids_train_yes = [k for k in ids_train if y_all[k][t1][0] >= THR_YES]
ids_val_yes   = [k for k in ids_val   if y_all[k][t1][0] >= THR_YES]
print(f"T*.1 (todas): train={len(ids_train)}  val={len(ids_val)}")
print(f"T*.2/3 (YES): train={len(ids_train_yes)}  val={len(ids_val_yes)}")

def task_train_ids(task): return ids_train if task_kind(task) == "bin" else ids_train_yes
def task_val_ids(task):   return ids_val   if task_kind(task) == "bin" else ids_val_yes

DEFAULTS = dict(epochs=5, batch_size=(8 if DEVICE == "cpu" else 16),
                lr_head=5e-4, lr_backbone=1e-5, warmup_frac=0.1, grad_clip=1.0,
                weight_decay=0.01, p_drop=0.2, sensor_hidden=256, sensor_out=128,
                fusion_hidden=512, num_heads=8)


In [ ]:
# === 10.2 · train_one_task ==================================================
def train_one_task(task, ckpt_name, m_gate=None, hp=None, trial=None, verbose=True):
    hp = {**DEFAULTS, **(hp or {})}
    ids_t = task_train_ids(task)
    loader_t = DataLoader(ExistDataset(ids_t), batch_size=hp["batch_size"], shuffle=True,
                          num_workers=0, collate_fn=collate, pin_memory=(DEVICE == "cuda"))
    model = MODEL_CLS[task](sensor_hidden=hp["sensor_hidden"], sensor_out=hp["sensor_out"],
                            fusion_hidden=hp["fusion_hidden"], num_heads=hp["num_heads"],
                            p_drop=hp["p_drop"], freeze_backbone=True).to(DEVICE)
    bb, hd = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad: continue
        (bb if n.startswith("text_encoder.") else hd).append(p)
    opt = torch.optim.AdamW([{"params": hd, "lr": hp["lr_head"]},
                             {"params": bb, "lr": hp["lr_backbone"]}], weight_decay=hp["weight_decay"])
    total = len(loader_t) * hp["epochs"]
    sched = get_linear_schedule_with_warmup(opt, int(total * hp["warmup_frac"]), total)
    fn = LOSS_FN[task]; gold = GOLD_VAL[task]
    history = {"train_loss": [], "f1": [], "icm_soft": [], "icm_soft_norm": []}
    best_metric, ckpt_path = -float("inf"), WORK_DIR / "ckpt" / ckpt_name
    for epoch in range(1, hp["epochs"] + 1):
        model.train(); running, seen = 0.0, 0
        pbar = tqdm(loader_t, desc=f"[T{task}] ep{epoch}/{hp['epochs']}", disable=not verbose)
        for batch in pbar:
            b = {k: (v.to(DEVICE) if isinstance(v, torch.Tensor) else v) for k, v in batch.items()}
            loss = fn(model(b), b)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), hp["grad_clip"])
            opt.step(); sched.step()
            n = b["input_ids"].size(0); running += loss.item() * n; seen += n
            pbar.set_postfix(loss=f"{running/max(1,seen):.4f}")
        train_loss = running / max(1, seen)
        f1 = evaluate_proxy(model, loader_val_full, task)
        try:
            icm = evaluate_pyevall(task, model, m_gate, loader_val_full, gold, hierarchical_gate=True)
        except Exception as e:
            if verbose: print(f"  [WARN] PyEvALL: {e}")
            icm = {"ICMSoft": float("nan"), "ICMSoftNorm": float("nan")}
        history["train_loss"].append(train_loss); history["f1"].append(f1)
        history["icm_soft"].append(icm.get("ICMSoft", float("nan")))
        history["icm_soft_norm"].append(icm.get("ICMSoftNorm", float("nan")))
        if verbose:
            print(f"  ep{epoch} train={train_loss:.4f} F1={f1:.3f} "
                  f"ICM-Soft={icm.get('ICMSoft', float('nan')):.4f} "
                  f"ICM-Soft-Norm={icm.get('ICMSoftNorm', float('nan')):.4f}")
        cur = icm.get("ICMSoftNorm", float("nan"))
        if trial is not None:
            trial.report(cur if cur == cur else -1.0, step=epoch)
            if trial.should_prune():
                if verbose: print(f"  [Optuna] trial {trial.number} podado ep{epoch}")
                raise optuna.TrialPruned()
        if cur == cur and cur > best_metric:
            best_metric = cur
            torch.save({"model": model.state_dict(), "task": task, "epoch": epoch,
                        "icm_soft": icm.get("ICMSoft"), "icm_soft_norm": cur, "hp": dict(hp)}, ckpt_path)
            if verbose: print(f"  >> mejor ckpt {ckpt_path.name} (ICM-Soft-Norm={cur:.4f})")
        gc.collect()
        if DEVICE == "cuda": torch.cuda.empty_cache()
    return model, history, best_metric

def load_model(task, ckpt_name):
    ck = torch.load(WORK_DIR / "ckpt" / ckpt_name, map_location=DEVICE, weights_only=False)
    hp = {**DEFAULTS, **(ck.get("hp") or {})}
    m = MODEL_CLS[task](sensor_hidden=hp["sensor_hidden"], sensor_out=hp["sensor_out"],
                        fusion_hidden=hp["fusion_hidden"], num_heads=hp["num_heads"],
                        p_drop=hp["p_drop"], freeze_backbone=True).to(DEVICE)
    m.load_state_dict(ck["model"]); m.eval()
    return m, ck

print("train_one_task + load_model definidos.")


## 11 · Optuna — TPE + MedianPruner

Optimiza cada subtarea maximizando el **ICM-Soft-Norm** en val. Búsqueda **secuencial**: T*.1 →
se reentrena su mejor config como *gate* → T*.2 y T*.3 con gating. Todas las combos
`fusion_hidden × num_heads` son divisibles (sin errores de dimensión). `BEST_HP` se persiste.


In [ ]:
# === 11.1 · Setup Optuna ====================================================
try:
    import optuna
except ImportError:
    import subprocess as _sp
    print("Instalando optuna..."); _sp.check_call([sys.executable, "-m", "pip", "install", "-q", "optuna"])
    import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
optuna.logging.set_verbosity(optuna.logging.WARNING)
print("Optuna:", optuna.__version__)

HPO_N_TRIALS = 20
HPO_EPOCHS   = 4
BEST_HP, OPTUNA_STUDIES = {}, {}

def _suggest_hp(trial):
    return {
        "lr_head":       trial.suggest_float("lr_head", 1e-4, 5e-3, log=True),
        "lr_backbone":   trial.suggest_float("lr_backbone", 1e-6, 1e-4, log=True),
        "warmup_frac":   trial.suggest_float("warmup_frac", 0.0, 0.2),
        "weight_decay":  trial.suggest_float("weight_decay", 1e-4, 1e-1, log=True),
        "grad_clip":     trial.suggest_float("grad_clip", 0.5, 2.0),
        "p_drop":        trial.suggest_float("p_drop", 0.1, 0.4),
        "sensor_hidden": trial.suggest_categorical("sensor_hidden", [128, 256, 384]),
        "sensor_out":    trial.suggest_categorical("sensor_out", [64, 128, 192]),
        "fusion_hidden": trial.suggest_categorical("fusion_hidden", [256, 384, 512, 768]),
        "num_heads":     trial.suggest_categorical("num_heads", [4, 8]),
        "batch_size":    trial.suggest_categorical("batch_size", [8, 16, 32]),
        "epochs":        HPO_EPOCHS,
    }

def run_optuna_for_task(task, n_trials=None, m_gate=None):
    n_trials = n_trials or HPO_N_TRIALS
    def objective(trial):
        _, _, best = train_one_task(task, ckpt_name=f"_optuna_{task}.pt", m_gate=m_gate,
                                    hp=_suggest_hp(trial), trial=trial, verbose=False)
        return best if (best == best and best > -float("inf")) else -1.0
    sampler = TPESampler(seed=SEED, multivariate=True, n_startup_trials=5)
    pruner  = MedianPruner(n_startup_trials=5, n_warmup_steps=1, interval_steps=1)
    study = optuna.create_study(direction="maximize", sampler=sampler, pruner=pruner,
                                study_name=f"{MODALITY}_{task}")
    print(f"\n{'='*70}\n  OPTUNA T{task}  ({n_trials} trials, {HPO_EPOCHS} epocas)\n{'='*70}")
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    BEST_HP[task] = study.best_params; OPTUNA_STUDIES[task] = study
    n_pr = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.PRUNED)
    print(f"\n  T{task}: mejor ICM-Soft-Norm={study.best_value:.4f} (podados {n_pr}/{len(study.trials)})")
    for k, v in study.best_params.items(): print(f"    {k:16s} = {v}")
    return study

print("Optuna configurado.")


In [ ]:
# === 11.2 · Busqueda secuencial T*.1 -> gate -> T*.2 / T*.3 =================
t1, t2, t3 = TASK_IDS
study_1 = run_optuna_for_task(t1, m_gate=None)
print(f"\n>> Reentrenando mejor T{t1} (gate)...")
_ = train_one_task(t1, ckpt_name=f"best_{t1}.pt", m_gate=None, hp=BEST_HP[t1], verbose=True)
m_gate, _ck = load_model(t1, f"best_{t1}.pt")
print(f"gate {t1} listo (ICM-Soft-Norm={_ck.get('icm_soft_norm', float('nan')):.4f})")
study_2 = run_optuna_for_task(t2, m_gate=m_gate)
study_3 = run_optuna_for_task(t3, m_gate=m_gate)


## 12 · Entrenamiento final con los mejores hiperparámetros + curvas

Reentrena las 3 subtareas con `BEST_HP` (más épocas) y guarda `best_*.pt` + `BEST_HP.json`.


In [ ]:
# === 12.1 · Entrenamiento final ============================================
FINAL_EPOCHS = 5
models, hists, bests = {}, {}, {}
for _t in TASK_IDS:
    hp = {**BEST_HP.get(_t, {}), "epochs": FINAL_EPOCHS}
    gate = None if task_kind(_t) == "bin" else m_gate
    print(f"\n{'='*70}\n  FINAL T{_t}\n{'='*70}")
    models[_t], hists[_t], bests[_t] = train_one_task(_t, ckpt_name=f"best_{_t}.pt",
                                                      m_gate=gate, hp=hp, verbose=True)
    if task_kind(_t) == "bin":
        m_gate, _ = load_model(_t, f"best_{_t}.pt")
json.dump(BEST_HP, open(WORK_DIR / "ckpt" / "BEST_HP.json", "w", encoding="utf-8"),
          ensure_ascii=False, indent=2)
print("\n=== Resumen ICM-Soft-Norm ===")
for _t in TASK_IDS: print(f"  T{_t}: {bests[_t]:.4f}")


In [ ]:
# === 12.2 · Curvas ==========================================================
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, _t in zip(axes, TASK_IDS):
    h = hists[_t]; ep = range(1, len(h["train_loss"]) + 1)
    ax.plot(ep, h["train_loss"], "-o", label="train loss", color="tab:gray")
    ax2 = ax.twinx()
    ax2.plot(ep, h["icm_soft_norm"], "-s", label="ICM-Soft-Norm", color="tab:red")
    ax2.plot(ep, h["f1"], "-^", label="F1", color="tab:blue", alpha=0.6)
    ax.set_title(f"T{_t}"); ax.set_xlabel("epoca"); ax.set_ylabel("loss")
    ax2.set_ylabel("ICM-Soft-Norm / F1")
    lns = ax.get_lines() + ax2.get_lines()
    ax.legend(lns, [l.get_label() for l in lns], fontsize=7, loc="center right")
    ax.grid(alpha=0.3)
plt.suptitle(f"Entrenamiento - {MODALITY} (cross-attention)")
plt.tight_layout()
plt.savefig(WORK_DIR / "figs" / "train_curves.png", dpi=120, bbox_inches="tight")
plt.show()


## 13 · Refit 100% + predicciones test (submission)

Reentrena con train+val (100%) usando `BEST_HP`, aplica el mismo preprocesado fisiológico al test
(ajustado sobre todo el dataset) y genera submissions PyEvALL con gating jerárquico. `run_id=2`.


In [ ]:
# === 13.1 · Refit preproc sobre TODO el dataset =============================
ids_all = ids_train + ids_val
if PHYSIO_KIND == "matrix":
    Xall = np.stack([PHYSIO_RAW[k] for k in ids_all]); Mall = np.stack([PHYSIO_MASK[k] for k in ids_all])
    _valid = Mall.reshape(-1) > 0
    knn_full = KNNImputer(n_neighbors=5, weights="distance")
    knn_full.fit(Xall.reshape(-1, FEAT_DIM)[_valid])
    _imp = lambda X: knn_full.transform(X.reshape(-1, FEAT_DIM)).reshape(X.shape)
    Xall_imp = _imp(Xall)
    scalers_full = {}
    for mod in ("ET", "HR", "EEG"):
        cols = np.where(MOD_OF == mod)[0]
        scalers_full[mod] = StandardScaler().fit(Xall_imp.reshape(-1, FEAT_DIM)[_valid][:, cols])
    def _std_full(X):
        flat = X.reshape(-1, FEAT_DIM).copy()
        for mod in ("ET", "HR", "EEG"):
            cols = np.where(MOD_OF == mod)[0]
            flat[:, cols] = scalers_full[mod].transform(flat[:, cols])
        return flat.reshape(X.shape).astype(np.float32)
    def physio_transform_full(raw_vec, mask): return _std_full(_imp(raw_vec[None]))[0], mask
else:
    Xall = np.stack([PHYSIO_RAW[k] for k in ids_all])
    mu_full = Xall.mean(axis=0).astype(np.float32); sigma_full = (Xall.std(axis=0) + 1e-8).astype(np.float32)
    def physio_transform_full(raw_vec, mask): return ((raw_vec - mu_full) / sigma_full).astype(np.float32), None
print(f"Refit preproc sobre {len(ids_all)} instancias listo.")


In [ ]:
# === 13.2 · train_full (refit 100%) =========================================
def train_full(task, m_gate=None, hp=None, n_epochs=None, verbose=True):
    hp = {**DEFAULTS, **(BEST_HP.get(task, {})), **(hp or {})}
    if n_epochs: hp["epochs"] = n_epochs
    ids_t = ids_all if task_kind(task) == "bin" else [k for k in ids_all if y_all[k][TASK_IDS[0]][0] >= THR_YES]
    loader = DataLoader(ExistDataset(ids_t), batch_size=hp["batch_size"], shuffle=True,
                        num_workers=0, collate_fn=collate, pin_memory=(DEVICE == "cuda"))
    model = MODEL_CLS[task](sensor_hidden=hp["sensor_hidden"], sensor_out=hp["sensor_out"],
                            fusion_hidden=hp["fusion_hidden"], num_heads=hp["num_heads"],
                            p_drop=hp["p_drop"], freeze_backbone=True).to(DEVICE)
    bb, hd = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad: continue
        (bb if n.startswith("text_encoder.") else hd).append(p)
    opt = torch.optim.AdamW([{"params": hd, "lr": hp["lr_head"]},
                             {"params": bb, "lr": hp["lr_backbone"]}], weight_decay=hp["weight_decay"])
    total = len(loader) * hp["epochs"]
    sched = get_linear_schedule_with_warmup(opt, int(total * hp["warmup_frac"]), total)
    fn = LOSS_FN[task]
    for epoch in range(1, hp["epochs"] + 1):
        model.train(); running, seen = 0.0, 0
        for batch in tqdm(loader, desc=f"[REFIT T{task}] ep{epoch}/{hp['epochs']}", disable=not verbose):
            b = {k: (v.to(DEVICE) if isinstance(v, torch.Tensor) else v) for k, v in batch.items()}
            loss = fn(model(b), b)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), hp["grad_clip"])
            opt.step(); sched.step()
            n = b["input_ids"].size(0); running += loss.item() * n; seen += n
        if verbose: print(f"    ep{epoch} loss={running/max(1,seen):.4f}")
    ckpt = WORK_DIR / "ckpt" / f"FULL_{task}.pt"
    torch.save({"model": model.state_dict(), "task": task, "hp": dict(hp)}, ckpt)
    if verbose: print(f"  FULL ckpt: {ckpt.name}")
    del opt; gc.collect()
    if DEVICE == "cuda": torch.cuda.empty_cache()
    return model
print("train_full definida.")


In [ ]:
# === 13.3 · Cargar test + preprocesar =======================================
TEST_JSON = next((p for p in CFG["test_json_candidates"] if p.exists()), None)
if TEST_JSON is None:
    print("[!] No se encontro TEST_JSON. Define CFG['test_json_candidates'] o salta la submission.")
    ids_test = []
else:
    print("TEST_JSON:", TEST_JSON)
    with open(TEST_JSON, "r", encoding="utf-8") as f:
        DATA_TEST = json.load(f)
    for k, v in DATA_TEST.items(): DATA[str(k)] = v
    ids_test = [str(k) for k in DATA_TEST.keys()]
    print(f"Test: {len(ids_test)} instancias")
    PHYSIO_TEST, PHYSIO_TEST_MASK = {}, {}
    for k in ids_test:
        if PHYSIO_KIND == "matrix":
            X, m = _sensor_matrix(DATA[k]); Xz, _ = physio_transform_full(X, m)
            PHYSIO_TEST[k] = Xz; PHYSIO_TEST_MASK[k] = m
        else:
            v = _flat_vector(DATA[k]); Xz, _ = physio_transform_full(v, None); PHYSIO_TEST[k] = Xz
    class TestDataset(Dataset):
        def __init__(self, ids): self.ids = ids
        def __len__(self): return len(self.ids)
        def __getitem__(self, idx):
            k = self.ids[idx]
            enc = tokenizer(build_text(DATA[k], VLM.get(k)), truncation=True,
                            max_length=MAX_LEN, padding="max_length", return_tensors="pt")
            item = {"id_EXIST": k, "input_ids": enc["input_ids"].squeeze(0),
                    "attention_mask": enc["attention_mask"].squeeze(0),
                    "physio": torch.from_numpy(np.ascontiguousarray(PHYSIO_TEST[k]))}
            if PHYSIO_KIND == "matrix":
                item["physio_mask"] = torch.from_numpy(np.ascontiguousarray(PHYSIO_TEST_MASK[k]))
            return item
    def collate_test(batch):
        out = {"id_EXIST": [b["id_EXIST"] for b in batch]}
        keys = ["input_ids", "attention_mask", "physio"] + (["physio_mask"] if PHYSIO_KIND == "matrix" else [])
        for k in keys: out[k] = torch.stack([b[k] for b in batch], dim=0)
        return out
    loader_test = DataLoader(TestDataset(ids_test), batch_size=16, shuffle=False,
                             num_workers=0, collate_fn=collate_test, pin_memory=(DEVICE == "cuda"))
    print("Test loader listo.")


In [ ]:
# === 13.4 · Refit 3 subtareas + submissions =================================
TEAM_NAME = "MAIN_STRATEGY"; RUN_ID = 2
if not ids_test:
    print("[!] Sin test cargado; define TEST_JSON.")
else:
    t1, t2, t3 = TASK_IDS
    full_1 = train_full(t1, m_gate=None, n_epochs=FINAL_EPOCHS)
    full_2 = train_full(t2, m_gate=full_1, n_epochs=FINAL_EPOCHS)
    full_3 = train_full(t3, m_gate=full_1, n_epochs=FINAL_EPOCHS)
    full_models = {t1: full_1, t2: full_2, t3: full_3}
    @torch.no_grad()
    def _infer(task, model, gate):
        model.eval()
        if gate is not None: gate.eval()
        kind = task_kind(task); out = {}
        for batch in tqdm(loader_test, desc=f"T{task} test", leave=False):
            b = {k: (v.to(DEVICE) if isinstance(v, torch.Tensor) else v) for k, v in batch.items()}
            o = model(b)
            p = (F.softmax(o["logits"], -1) if kind != "multi" else torch.sigmoid(o["logits"])).cpu().numpy()
            pg = None if kind == "bin" else F.softmax(gate(b)["logits"], -1).cpu().numpy()
            for i, vid in enumerate(b["id_EXIST"]):
                out[str(vid)] = (p[i], None if pg is None else pg[i])
        return out
    def _build_sub(task, preds):
        kind = task_kind(task); rows = []
        for vid in ids_test:
            p, pg = preds[vid]
            if kind == "bin":
                v = {"YES": float(p[0]), "NO": float(p[1])}
            elif kind == "tri":
                v = ({"NO": 1.0, "DIRECT": 0.0, "JUDGEMENTAL": 0.0} if (pg is not None and pg[1] > pg[0])
                     else {"NO": float(p[0]), "DIRECT": float(p[1]), "JUDGEMENTAL": float(p[2])})
            else:
                if pg is not None and pg[1] > pg[0]:
                    v = {"NO": 1.0, "IDEOLOGICAL-INEQUALITY": 0.0, "STEREOTYPING-DOMINANCE": 0.0,
                         "OBJECTIFICATION": 0.0, "SEXUAL-VIOLENCE": 0.0, "MISOGYNY-NON-SEXUAL-VIOLENCE": 0.0}
                else:
                    v = {"NO": float(p[0]), "IDEOLOGICAL-INEQUALITY": float(p[1]),
                         "STEREOTYPING-DOMINANCE": float(p[2]), "OBJECTIFICATION": float(p[3]),
                         "SEXUAL-VIOLENCE": float(p[4]), "MISOGYNY-NON-SEXUAL-VIOLENCE": float(p[5])}
            rows.append({"test_case": PYEVALL_TEST_CASE, "id": str(vid), "value": v})
        return rows
    save_dir = WORK_DIR / "submissions" / f"exist2026_{TEAM_NAME}"; save_dir.mkdir(parents=True, exist_ok=True)
    for _t in TASK_IDS:
        gate = None if task_kind(_t) == "bin" else full_1
        sub = _build_sub(_t, _infer(_t, full_models[_t], gate))
        out_path = save_dir / f"task{_t[0]}_{_t[1]}_soft_{TEAM_NAME}_{RUN_ID}"
        json.dump(sub, open(out_path, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
        print(f"  guardado {out_path.name} ({len(sub)} preds)")
    print("\nSubmissions en", save_dir)
